# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one client's one content item's performance on one calendar day — the grain is `report_date x client_id x content_id` in `fact_content_daily_performance`.

Time window: `month = 2026-03` (a mid-panel month, per the panel warning that `_sample` is actually June 2026, the sealed final month — not touched here).

Target/proxy for this slice: whether a content item's impressions fell from the first half of the month to the second half (`sum(impressions) days 16-31 < 0.8 x sum(impressions) days 1-15`, among items with real first-half volume). This is a real forward-looking proxy built from two actual time slices inside one panel month.

In [ ]:
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

print(f"HF_TOKEN loaded: {HF_TOKEN is not None}")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")

base = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (first-half only, knowable at decision time):**
- `avg_daily_impressions_first_half` — avg of `impressions`, days 1-15
- `avg_position_first_half` — avg of `gsc_avg_position`, days 1-15
- `days_with_sessions_first_half` — count of distinct days with `sessions > 0`, days 1-15
- `ctr_first_half` — `sum(clicks)/sum(impressions)`, days 1-15
- `content_age_days` (from `dim_content`) — static attribute, true on any day

**Label (second-half only, allowed to see the "future" relative to the features):**
- `second_half_impressions` — `sum(impressions)`, days 16-31 -> used only to derive `declining`, never as a feature

**Context (not modeled, used to interpret/filter):**
- `client_id`, `content_id` — grouping/joining only, never features
- `ga4_data_available` — filter flag, not a feature
- `dim_clients.ga4_data_start` — checked before trusting any GA4 column in this window

**Excluded, with why:**
- `fact_content_query_90d` — its 90-day window overlaps this month's boundaries; mixing it in risks the window-alignment leak the data skill warns about. One table, one month keeps this contract clean.

In [3]:
# nothing to execute for this section.

## 3. Verify it with queries (grain, counts, missing values, windows) + five features + the leakage trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query A — grain check**, **Query B — row count + date span**, **Query C — availability with `IS TRUE`** are in the code cell below, followed by the five-feature frame and the deliberate-leak experiment.

*(Fill in after running: "Query A returned [N] rows — [empty / not empty, meaning grain holds / breaks]. Query B: [N] rows spanning [min date] to [max date]. Query C: [N] of [N] total rows have `ga4_data_available IS TRUE`.")*

In [4]:
# Query A: grain check
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{base}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain check (should be empty):")
print(grain_check)

# Query B: slice size and date span
size_span = con.execute(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{base}')
""").df()
print("\nSlice size and date span:")
print(size_span)

# Query C: availability, filtered with IS TRUE
availability = con.execute(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
    FROM read_parquet('{base}')
""").df()
print("\nAvailability (IS TRUE filter):")
print(availability)

# Five features
features = con.execute(f"""
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_impressions) AS avg_daily_impressions_first_half,
           AVG(gsc_avg_position) AS avg_position_first_half,
           COUNT(DISTINCT CASE WHEN (sessions_ai + sessions_paid + sessions_direct + sessions_social + sessions_organic) > 0 THEN report_date END) AS days_with_sessions_first_half,
           SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr_first_half
    FROM read_parquet('{base}')
    WHERE ga4_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1,2
""").df()
print("\nFive-feature frame (first 10 rows):")
print(features.head(10))
# available when: every column above uses only days 1-15, strictly before the label window.
# content_age_days would be the 5th feature, joined from dim_content (static, always knowable).

# The trap: deliberate leak, then removal
label = con.execute(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS second_half_impressions
    FROM read_parquet('{base}')
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
      AND ga4_data_available IS TRUE
    GROUP BY 1,2
""").df()

merged = features.merge(label, on=["client_hash_id","content_hash_id"])
merged["declining"] = (merged.second_half_impressions < 0.8 * merged.avg_daily_impressions_first_half * 15).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# LEAKY: second_half_impressions is literally what the label is derived from
X_leaky = merged[["avg_daily_impressions_first_half","avg_position_first_half","second_half_impressions"]].fillna(0).astype(float)
y = merged["declining"]
leaky_score = roc_auc_score(y, LogisticRegression(max_iter=1000).fit(X_leaky, y).predict_proba(X_leaky)[:,1])
print(f"\nLeaky AUC (watch it jump toward 1.0): {leaky_score:.3f}")

# HONEST: drop the leaked column, keep this number
X_honest = merged[["avg_daily_impressions_first_half","avg_position_first_half"]].fillna(0).astype(float)
honest_score = roc_auc_score(y, LogisticRegression(max_iter=1000).fit(X_honest, y).predict_proba(X_honest)[:,1])
print(f"Honest AUC (the one to keep): {honest_score:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []

Slice size and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability (IS TRUE filter):
   total_rows  available_rows
0     9841378        413966.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Five-feature frame (first 10 rows):
            client_hash_id           content_hash_id  \
0  client_9958f0a7ae1df715  content_eb0aeedbcfaf2712   
1  client_9958f0a7ae1df715  content_651b8ba180f9beff   
2  client_9958f0a7ae1df715  content_1f39e904c7351258   
3  client_9958f0a7ae1df715  content_5b619639be88dea7   
4  client_9958f0a7ae1df715  content_3a3e193ec1e76e3b   
5  client_9958f0a7ae1df715  content_a46986d3796592f9   
6  client_9958f0a7ae1df715  content_1021a22410889b2a   
7  client_9958f0a7ae1df715  content_232aa83d13706e4b   
8  client_9958f0a7ae1df715  content_cc1ce98b4eb10e0c   
9  client_9958f0a7ae1df715  content_f216815b76a1bdb8   

   avg_daily_impressions_first_half  avg_position_first_half  \
0                         25.000000                 6.683673   
1                          8.000000                17.435623   
2                         29.666667                12.800961   
3                         13.400000                 7.822601   
4                         

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Leaky AUC (watch it jump toward 1.0): 1.000
Honest AUC (the one to keep): 0.812


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice only covers clients with `ga4_data_available = TRUE` for March 2026 — a third of clients have little or no usable GA4 history, so this feature frame systematically excludes them; any pattern found here may not generalize to that excluded third. It also can't say anything about query-level behavior (`fact_content_query_90d` was deliberately excluded), and the "declining" proxy is a within-month comparison, not a validated long-term outcome.

In [10]:
# nothing to execute for this section.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.